# 01 — Target Definition & Data Cleaning

**Project:** `ecommerce-delivery-delay-prediction`  
**Phase:** 2 — Data Preparation & Understanding

## Goals

Notebook này có các mục tiêu:

1. Xác định modeling population.
2. Điều tra các order bất thường liên quan đến target.
3. Xác định prediction point hợp lệ.
4. Tạo target `late_delivery`.
5. Kiểm tra target distribution.
6. Định nghĩa cleaning rules.
7. Kiểm tra data quality invariants.
8. Xuất clean order-level dataset vào `data/interim/`.

> Notebook này chưa thực hiện feature engineering hoặc model training.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Project paths

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Không tìm thấy project root chứa data/raw."
    )


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_METRICS_DIR = PROJECT_ROOT / "reports" / "metrics"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)

PROJECT_ROOT: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction
RAW_DIR: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/raw
INTERIM_DIR: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/interim


## 3. Load orders

In [3]:
orders = pd.read_csv(
    RAW_DIR / "olist_orders_dataset.csv"
)

print("Shape:", orders.shape)
display(orders.head())

Shape: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


## 4. Basic raw-data validation

In [4]:
assert len(orders) == 99_441
assert orders["order_id"].is_unique
assert orders["order_purchase_timestamp"].notna().all()
assert orders["order_estimated_delivery_date"].notna().all()

print("✓ Basic raw-order validations passed.")

✓ Basic raw-order validations passed.


## 5. Parse timestamps

In [5]:
TIMESTAMP_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_clean = orders.copy()

for col in TIMESTAMP_COLUMNS:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col],
        errors="coerce",
    )

display(
    orders_clean[TIMESTAMP_COLUMNS]
    .dtypes
    .rename("dtype")
    .to_frame()
)

,dtype
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


# 6. Modeling Population Investigation

Target của project là:

`late_delivery = actual delivery > estimated delivery`

Prediction được thực hiện tại:

`order_approved_at`

Vì vậy một order chỉ có thể thuộc modeling population nếu:

- order đã thực sự được delivered;
- prediction point `order_approved_at` tồn tại;
- actual delivery timestamp tồn tại;
- estimated delivery timestamp tồn tại.

## 6.1 Delivered orders

In [6]:
delivered_orders = orders_clean[
    orders_clean["order_status"].eq("delivered")
].copy()

print("Total raw orders:", len(orders_clean))
print("Delivered orders:", len(delivered_orders))
print(
    "Delivered percentage:",
    f"{len(delivered_orders) / len(orders_clean) * 100:.2f}%"
)

Total raw orders: 99441
Delivered orders: 96478
Delivered percentage: 97.02%


## 6.2 Delivered nhưng thiếu actual delivery

In [7]:
delivered_missing_actual = delivered_orders[
    delivered_orders[
        "order_delivered_customer_date"
    ].isna()
].copy()

print(
    "Delivered orders missing actual delivery:",
    len(delivered_missing_actual),
)

display(
    delivered_missing_actual[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
)

Delivered orders missing actual delivery: 8


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


Các order này không thể tạo target vì `order_delivered_customer_date` bị thiếu.

**Cleaning decision:** exclude khỏi modeling population.

## 6.3 Delivered nhưng thiếu approval timestamp

In [8]:
delivered_missing_approval = delivered_orders[
    delivered_orders["order_approved_at"].isna()
].copy()

print(
    "Delivered orders missing approval timestamp:",
    len(delivered_missing_approval),
)

display(
    delivered_missing_approval[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
)

Delivered orders missing approval timestamp: 14


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20


Prediction point của project là `order_approved_at`, nên các order thiếu timestamp này không thể mô phỏng đúng production prediction time.

**Cleaning decision:** exclude khỏi modeling population.

## 6.4 Kiểm tra overlap giữa các nhóm missing

In [9]:
missing_actual_ids = set(
    delivered_missing_actual["order_id"]
)

missing_approval_ids = set(
    delivered_missing_approval["order_id"]
)

overlap_ids = (
    missing_actual_ids
    & missing_approval_ids
)

print(
    "Orders missing both actual delivery and approval:",
    len(overlap_ids),
)

overlap_ids

Orders missing both actual delivery and approval: 0


set()

## 6.5 Canceled nhưng có actual delivery timestamp

In [10]:
canceled_with_delivery = orders_clean[
    orders_clean["order_status"].eq("canceled")
    & orders_clean[
        "order_delivered_customer_date"
    ].notna()
].copy()

print(
    "Canceled orders with actual delivery timestamp:",
    len(canceled_with_delivery),
)

display(
    canceled_with_delivery[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
        ]
    ]
)

Canceled orders with actual delivery timestamp: 6


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


Dù một số canceled order có actual delivery timestamp, chúng vẫn nằm ngoài modeling population vì project định nghĩa target trên các order đã hoàn tất delivery lifecycle với trạng thái `delivered`.

**Cleaning decision:** exclude các order có `order_status != "delivered"`.

## 6.6 Invalid promised delivery window

Prediction được thực hiện tại `order_approved_at`.

Tại thời điểm này, `order_estimated_delivery_date` phải là một delivery commitment hợp lệ trong tương lai. Vì vậy:

`order_estimated_delivery_date < order_approved_at`

được xem là data-quality anomaly và sẽ bị loại khỏi modeling population.

In [11]:
invalid_promised_window = (
    orders_clean["order_approved_at"].notna()
    &
    (
        orders_clean["order_estimated_delivery_date"]
        <
        orders_clean["order_approved_at"]
    )
)

print(
    "Orders with estimated delivery before approval:",
    int(invalid_promised_window.sum()),
)

display(
    orders_clean.loc[
        invalid_promised_window,
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_estimated_delivery_date",
            "order_delivered_customer_date",
        ],
    ]
)

Orders with estimated delivery before approval: 12


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_estimated_delivery_date,order_delivered_customer_date
10071,809a282bbd5dbcabb6f2f724fca862ec,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,2016-09-30,NaT
43418,5e981569f5835c96e4b288363b3b8f63,delivered,2018-07-30 16:25:37,2018-08-02 13:43:13,2018-08-02,2018-08-01 15:17:54
43697,0a5c74ccc786ced7903270de9d6c170a,unavailable,2018-01-18 23:14:36,2018-02-20 12:05:54,2018-02-14,NaT
47552,1612081119e8f23745698ad3367cc14b,unavailable,2016-10-05 18:06:48,2017-04-11 15:17:38,2016-11-17,NaT
50623,e73fe43cdcd166f7f0c6e3c2bf11a917,delivered,2018-08-09 18:06:43,2018-08-20 15:57:28,2018-08-16,2018-08-15 15:46:38
55230,69fd1a4dd8b96b9ab0dc82b95be21efd,unavailable,2018-07-31 09:55:46,2018-08-05 23:30:50,2018-08-03,NaT
55353,9675440ebf61a1a3482cc6308e3ebd28,delivered,2018-08-18 23:35:23,2018-08-24 22:05:08,2018-08-23,2018-08-28 17:37:30
57637,394d17c2b71a726e205caaeee3d2aa3d,delivered,2018-08-03 14:56:26,2018-08-08 23:31:22,2018-08-08,2018-08-07 19:28:25
61544,6dcf0aeb8b1eb4021c26e1d0e9394979,delivered,2018-08-09 20:37:34,2018-08-20 15:59:18,2018-08-15,2018-08-15 19:06:29
62293,2e5dc86c8c4aa663549caf5e31de840d,processing,2017-02-03 00:04:49,2017-04-04 10:56:48,2017-03-31,NaT


# 7. Define Eligibility Rules

In [12]:
is_delivered = (
    orders_clean["order_status"] == "delivered"
)

has_purchase_timestamp = (
    orders_clean[
        "order_purchase_timestamp"
    ].notna()
)

has_prediction_point = (
    orders_clean[
        "order_approved_at"
    ].notna()
)

has_actual_delivery = (
    orders_clean[
        "order_delivered_customer_date"
    ].notna()
)

has_estimated_delivery = (
    orders_clean[
        "order_estimated_delivery_date"
    ].notna()
)

has_valid_promised_window = (
    orders_clean["order_approved_at"].notna()
    &
    (
        orders_clean["order_estimated_delivery_date"]
        >=
        orders_clean["order_approved_at"]
    )
)


## 7.1 Final eligible mask

In [13]:
eligible_mask = (
    is_delivered
    & has_purchase_timestamp
    & has_prediction_point
    & has_actual_delivery
    & has_estimated_delivery
    & has_valid_promised_window
)

modeling_orders = (
    orders_clean.loc[eligible_mask]
    .copy()
    .reset_index(drop=True)
)

print(
    "Final modeling population:",
    len(modeling_orders),
)

print(
    "Retained percentage:",
    f"{len(modeling_orders) / len(orders_clean) * 100:.2f}%"
)

Final modeling population: 96450
Retained percentage: 96.99%


## 7.2 Modeling population funnel

In [14]:
population_funnel = pd.DataFrame(
    {
        "step": [
            "Raw orders",
            "Delivered",
            "Delivered + purchase timestamp",
            "Delivered + prediction point",
            "Delivered + actual delivery",
            "Delivered + estimated delivery",
            "Delivered + valid promised window",
            "Final eligible population",
        ],
        "orders": [
            len(orders_clean),
            int(is_delivered.sum()),
            int(
                (
                    is_delivered
                    & has_purchase_timestamp
                ).sum()
            ),
            int(
                (
                    is_delivered
                    & has_purchase_timestamp
                    & has_prediction_point
                ).sum()
            ),
            int(
                (
                    is_delivered
                    & has_purchase_timestamp
                    & has_prediction_point
                    & has_actual_delivery
                ).sum()
            ),
            int(
                (
                    is_delivered
                    & has_purchase_timestamp
                    & has_prediction_point
                    & has_actual_delivery
                    & has_estimated_delivery
                ).sum()
            ),
            int(
                (
                    is_delivered
                    & has_purchase_timestamp
                    & has_prediction_point
                    & has_actual_delivery
                    & has_estimated_delivery
                    & has_valid_promised_window
                ).sum()
            ),
            int(eligible_mask.sum()),
        ],
    }
)

population_funnel["retained_pct"] = (
    population_funnel["orders"]
    / len(orders_clean)
    * 100
)

display(population_funnel)


,step,orders,retained_pct
0,Raw orders,99441,100.0000
1,Delivered,96478,97.0203
2,Delivered + purchase timestamp,96478,97.0203
3,Delivered + prediction point,96464,97.0063
4,Delivered + actual delivery,96456,96.9982
5,Delivered + estimated delivery,96456,96.9982
6,Delivered + valid promised window,96450,96.9922
7,Final eligible population,96450,96.9922


# 8. Target Creation

Target:

- `0`: order delivered on or before estimated delivery timestamp.
- `1`: order delivered after estimated delivery timestamp.

`order_delivered_customer_date` được dùng để tạo target nhưng **không được sử dụng làm model feature**.

In [15]:
modeling_orders["late_delivery"] = (
    modeling_orders[
        "order_delivered_customer_date"
    ]
    >
    modeling_orders[
        "order_estimated_delivery_date"
    ]
).astype("int8")

## 8.1 Kiểm tra target samples

In [16]:
display(
    modeling_orders[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "late_delivery",
        ]
    ].sample(
        10,
        random_state=42,
    )
)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,late_delivery
29561,58acbb7315ddfcd40497860d21ab1662,2018-03-08 14:37:27,2018-03-12,0
25462,0dbfe0ba49bde79404577f3b85d0058e,2017-12-14 17:28:41,2017-12-20,0
86016,8ca358a9578fb7919efe79785693d479,2017-05-17 10:04:59,2017-05-30,0
26587,9a1be11e9a732a340c3bec3b390debeb,2018-07-09 23:16:49,2018-07-25,0
3016,035d66b23a4bba74f8119f9ebe3458ee,2018-06-06 14:12:25,2018-06-05,1
46651,a57f69a30b656e82f1c0a04f0f644e34,2018-08-09 00:18:33,2018-08-15,0
86244,0543af37e49147c9e0293a6d700917cb,2017-06-09 16:29:28,2017-06-22,0
64797,519440577b644a6822242fe9f13b31e8,2018-08-08 21:12:27,2018-08-09,0
73674,6aa3661755d14531db724d1af0dfbaa3,2018-04-11 21:41:12,2018-04-19,0
58448,6feb1191d85c6196e2dde76966fdb45a,2018-01-31 14:22:08,2018-02-16,0


## 8.2 Target distribution

In [17]:
target_summary = (
    modeling_orders["late_delivery"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

target_summary["pct"] = (
    target_summary["count"]
    / len(modeling_orders)
    * 100
)

target_summary.index = target_summary.index.map(
    {
        0: "on_time",
        1: "late",
    }
)

display(target_summary)

,count,pct
late_delivery,,
on_time,88627,91.8891
late,7823,8.1109


In [18]:
late_rate = (
    modeling_orders["late_delivery"].mean()
)

print(
    f"Late delivery rate: {late_rate:.2%}"
)

Late delivery rate: 8.11%


# 9. Analysis-only Delivery Delay

`delivery_delay_days` dùng actual delivery timestamp nên chỉ phục vụ:

- EDA
- target analysis
- error analysis

**Không được dùng làm model feature.**

In [19]:
modeling_orders["delivery_delay_days"] = (
    modeling_orders[
        "order_delivered_customer_date"
    ]
    -
    modeling_orders[
        "order_estimated_delivery_date"
    ]
).dt.total_seconds() / 86_400

display(
    modeling_orders[
        "delivery_delay_days"
    ].describe()
)

count   96,450.0000
mean       -11.1774
std         10.1839
min       -146.0161
25%        -16.2438
50%        -11.9460
75%         -6.3901
max        188.9751
Name: delivery_delay_days, dtype: float64

## 9.1 Validate target logic

In [20]:
assert (
    modeling_orders.loc[
        modeling_orders["late_delivery"] == 1,
        "delivery_delay_days",
    ]
    > 0
).all()

assert (
    modeling_orders.loc[
        modeling_orders["late_delivery"] == 0,
        "delivery_delay_days",
    ]
    <= 0
).all()

print("✓ Target logic validated.")

✓ Target logic validated.


# 10. Timestamp Anomaly Investigation

Dataset Understanding phát hiện hai anomaly đáng chú ý:

- carrier timestamp trước approval;
- customer delivery timestamp trước carrier timestamp.

Không tự động xóa các rows này trước khi hiểu tác động.

## 10.1 Carrier trước approval

In [21]:
carrier_before_approval_mask = (
    modeling_orders[
        "order_delivered_carrier_date"
    ].notna()
    &
    (
        modeling_orders[
            "order_delivered_carrier_date"
        ]
        <
        modeling_orders[
            "order_approved_at"
        ]
    )
)

carrier_before_approval = modeling_orders.loc[
    carrier_before_approval_mask
].copy()

print(
    "Carrier before approval:",
    len(carrier_before_approval),
)

Carrier before approval: 1345


In [22]:
carrier_before_approval[
    "approval_minus_carrier_hours"
] = (
    (
        carrier_before_approval[
            "order_approved_at"
        ]
        -
        carrier_before_approval[
            "order_delivered_carrier_date"
        ]
    )
    .dt.total_seconds()
    / 3600
)

display(
    carrier_before_approval[
        "approval_minus_carrier_hours"
    ].describe()
)

count   1,345.0000
mean       24.3880
std       115.7211
min         0.0058
25%         1.4089
50%        16.8189
75%        25.9214
max     4,109.2561
Name: approval_minus_carrier_hours, dtype: float64

In [23]:
display(
    carrier_before_approval[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "approval_minus_carrier_hours",
        ]
    ]
    .sort_values(
        "approval_minus_carrier_hours",
        ascending=False,
    )
    .head(20)
)

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,approval_minus_carrier_hours
25120,7c48bb55e8e4f7e56d412e9653db37bc,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,"4,109.2561"
14146,1fab4ac9d85079b3da72a11475ae1685,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,224.9300
44775,0184d4ddb259e1a4cfc2871888cf97b8,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,224.1903
95741,1378f9601350615613cc8832d6789c5d,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,219.9322
40349,8554cb37f7158cb0b082a841d24a4589,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,218.7625
11406,cf72398d0690f841271b695bbfda82d2,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,217.8661
82808,40de47dfa620d667117e4a6067b6e1ec,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,217.3611
30280,bc4854efd86d9f42140c951c595d20c1,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,217.1817
53646,77ca435b03fbf991e5027e3776e37885,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,217.1022
66260,580f3268bc2075c5961b2d2929c7a35b,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,202.0900


**Decision:** không auto-drop.

`order_delivered_carrier_date` là post-prediction information và sẽ không được dùng làm model feature.

## 10.2 Customer delivery trước carrier

In [24]:
delivery_before_carrier_mask = (
    modeling_orders[
        "order_delivered_carrier_date"
    ].notna()
    &
    (
        modeling_orders[
            "order_delivered_customer_date"
        ]
        <
        modeling_orders[
            "order_delivered_carrier_date"
        ]
    )
)

delivery_before_carrier = modeling_orders.loc[
    delivery_before_carrier_mask
].copy()

print(
    "Delivery before carrier:",
    len(delivery_before_carrier),
)

Delivery before carrier: 23


In [25]:
delivery_before_carrier[
    "carrier_minus_delivery_hours"
] = (
    (
        delivery_before_carrier[
            "order_delivered_carrier_date"
        ]
        -
        delivery_before_carrier[
            "order_delivered_customer_date"
        ]
    )
    .dt.total_seconds()
    / 3600
)

display(
    delivery_before_carrier[
        "carrier_minus_delivery_hours"
    ].describe()
)

count    23.0000
mean     78.4551
std      89.3081
min       0.3883
25%      23.3383
50%      39.8644
75%     128.7590
max     386.3081
Name: carrier_minus_delivery_hours, dtype: float64

In [26]:
display(
    delivery_before_carrier[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "carrier_minus_delivery_hours",
            "late_delivery",
        ]
    ]
)

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,carrier_minus_delivery_hours,late_delivery
6251,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,69.4172,0
9286,383aa8b2724fe452d9ccd9934a8c628b,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,26.9139,0
13107,cb1134f9010d242e9515ad1c78ec0c39,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,29.1428,0
14059,dceb62e8fa94b46006c9554fed743df0,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,144.2389,0
18719,5f9d46795c3126674e52becb3a1a517f,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,4.1836,0
20719,8c78d01de3a9009e23d6877a7cc9be20,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,17.8353,0
21861,b27af682321527a6349f1761eb3f360c,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,23.1053,0
24641,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,0.3883,0
24891,e37f11cae9985ca58f0b56f268720537,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,24.4642,0
26654,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,189.0950,0


## 10.3 Critical timestamp consistency

In [27]:
approval_before_purchase = (
    modeling_orders["order_approved_at"]
    <
    modeling_orders["order_purchase_timestamp"]
)

delivery_before_purchase = (
    modeling_orders[
        "order_delivered_customer_date"
    ]
    <
    modeling_orders[
        "order_purchase_timestamp"
    ]
)

estimated_before_purchase = (
    modeling_orders[
        "order_estimated_delivery_date"
    ]
    <
    modeling_orders[
        "order_purchase_timestamp"
    ]
)

print(
    "Approval before purchase:",
    approval_before_purchase.sum(),
)

print(
    "Actual delivery before purchase:",
    delivery_before_purchase.sum(),
)

print(
    "Estimated delivery before purchase:",
    estimated_before_purchase.sum(),
)

Approval before purchase: 0
Actual delivery before purchase: 0
Estimated delivery before purchase: 0


# 11. Cleaning Rules

In [28]:
cleaning_rules = pd.DataFrame(
    [
        {
            "rule": "order_status must be delivered",
            "action": "exclude otherwise",
            "reason": "Target represents completed delivery lifecycle.",
        },
        {
            "rule": "order_approved_at must exist",
            "action": "exclude if missing",
            "reason": "Required prediction point.",
        },
        {
            "rule": "actual delivery must exist",
            "action": "exclude if missing",
            "reason": "Required for target construction.",
        },
        {
            "rule": "estimated delivery must exist",
            "action": "exclude if missing",
            "reason": "Required for target construction.",
        },
        {
            "rule": "approval cannot precede purchase",
            "action": "exclude if violated",
            "reason": "Invalid lifecycle.",
        },
        {
            "rule": "actual delivery cannot precede purchase",
            "action": "exclude if violated",
            "reason": "Invalid lifecycle.",
        },
        {
            "rule": "carrier before approval",
            "action": "keep + document",
            "reason": (
                "Carrier timestamp is post-prediction and "
                "will not be used as feature."
            ),
        },
        {
            "rule": "delivery before carrier",
            "action": "investigate",
            "reason": "Potential timestamp logging anomaly.",
        },
        {
            "rule": "estimated delivery cannot precede approval",
            "action": "exclude if violated",
            "reason": (
                "Estimated delivery date must be valid at the "
                "order_approved_at prediction point."
            ),
        },
    ]
)

display(cleaning_rules)

,rule,action,reason
0,order_status must be delivered,exclude otherwise,Target represents completed delivery lifecycle.
1,order_approved_at must exist,exclude if missing,Required prediction point.
2,actual delivery must exist,exclude if missing,Required for target construction.
3,estimated delivery must exist,exclude if missing,Required for target construction.
4,approval cannot precede purchase,exclude if violated,Invalid lifecycle.
5,actual delivery cannot precede purchase,exclude if violated,Invalid lifecycle.
6,carrier before approval,keep + document,Carrier timestamp is post-prediction and will not be used as feature.
7,delivery before carrier,investigate,Potential timestamp logging anomaly.
8,estimated delivery cannot precede approval,exclude if violated,Estimated delivery date must be valid at the order_approved_at prediction point.


# 12. Data Quality Assertions

In [29]:
assert modeling_orders["order_id"].is_unique

assert modeling_orders[
    "order_status"
].eq("delivered").all()

assert modeling_orders[
    "order_purchase_timestamp"
].notna().all()

assert modeling_orders[
    "order_approved_at"
].notna().all()

assert modeling_orders[
    "order_delivered_customer_date"
].notna().all()

assert modeling_orders[
    "order_estimated_delivery_date"
].notna().all()

assert modeling_orders[
    "late_delivery"
].isin([0, 1]).all()

assert not (
    modeling_orders[
        "order_approved_at"
    ]
    <
    modeling_orders[
        "order_purchase_timestamp"
    ]
).any()

assert not (
    modeling_orders[
        "order_delivered_customer_date"
    ]
    <
    modeling_orders[
        "order_purchase_timestamp"
    ]
).any()

assert not (
    modeling_orders["order_estimated_delivery_date"]
    <
    modeling_orders["order_approved_at"]
).any()

print("✓ Promised delivery window is valid.")

print(
    "✓ All modeling population invariants passed."
)

✓ Promised delivery window is valid.
✓ All modeling population invariants passed.


# 13. Leakage Registry

In [30]:
FORBIDDEN_MODEL_FEATURES = {
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "delivery_delay_days",
}

FORBIDDEN_REVIEW_FEATURES = {
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
}

print("Order leakage columns:")
print(FORBIDDEN_MODEL_FEATURES)

print("\nReview leakage columns:")
print(FORBIDDEN_REVIEW_FEATURES)

Order leakage columns:
{'delivery_delay_days', 'order_delivered_carrier_date', 'order_delivered_customer_date'}

Review leakage columns:
{'review_comment_message', 'review_answer_timestamp', 'review_score', 'review_comment_title', 'review_creation_date'}


# 14. Export Interim Dataset

In [31]:
INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH = (
    INTERIM_DIR
    / "orders_cleaned.parquet"
)

modeling_orders.to_parquet(
    OUTPUT_PATH,
    index=False,
)

print(
    f"✓ Saved: {OUTPUT_PATH}"
)

print(
    f"Rows: {len(modeling_orders):,}"
)

✓ Saved: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/interim/orders_cleaned.parquet
Rows: 96,450


# 15. Export Reports

In [32]:
REPORTS_METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

population_funnel.to_csv(
    REPORTS_METRICS_DIR
    / "modeling_population_funnel.csv",
    index=False,
)

target_summary.to_csv(
    REPORTS_METRICS_DIR
    / "target_distribution.csv",
)

cleaning_rules.to_csv(
    REPORTS_METRICS_DIR
    / "cleaning_rules.csv",
    index=False,
)

print("✓ Reports exported.")

✓ Reports exported.


# 16. Final Summary

In [33]:
print("=" * 80)
print("TARGET & CLEANING SUMMARY")
print("=" * 80)

print(
    f"\nRaw orders:"
    f" {len(orders_clean):,}"
)

print(
    f"Delivered orders:"
    f" {is_delivered.sum():,}"
)

print(
    f"Final modeling population:"
    f" {len(modeling_orders):,}"
)

print(
    f"Retained:"
    f" {len(modeling_orders) / len(orders_clean):.2%}"
)

print(
    f"\nLate orders:"
    f" {modeling_orders['late_delivery'].sum():,}"
)

print(
    f"Late rate:"
    f" {modeling_orders['late_delivery'].mean():.2%}"
)

print(
    f"\nCarrier before approval:"
    f" {carrier_before_approval_mask.sum():,}"
)

print(
    f"Delivery before carrier:"
    f" {delivery_before_carrier_mask.sum():,}"
)

print(
    "\nOutput:",
    OUTPUT_PATH,
)

TARGET & CLEANING SUMMARY

Raw orders: 99,441
Delivered orders: 96,478
Final modeling population: 96,450
Retained: 96.99%

Late orders: 7,823
Late rate: 8.11%

Carrier before approval: 1,345
Delivery before carrier: 23

Output: /home/namdp/Documents/Projects/ecommerce-delivery-delay-prediction/data/interim/orders_cleaned.parquet


# 17. Conclusions

Sau khi Run All, sử dụng chính các output của notebook làm source of truth.

## Modeling population

The modeling population contains only orders that:

- have `order_status == "delivered"`;
- have a valid purchase timestamp;
- have a valid `order_approved_at` prediction point;
- have an actual customer delivery timestamp;
- have an estimated delivery timestamp;
- satisfy `order_estimated_delivery_date >= order_approved_at`.

The last rule removes invalid negative promised-delivery windows.

The exact final population is reported in `population_funnel`.

## Target

The target is:

`late_delivery = order_delivered_customer_date > order_estimated_delivery_date`

The exact target distribution is reported in `target_summary`.

Because the late class is the minority, later model evaluation will emphasize PR-AUC, Recall, Precision, and F1-score rather than Accuracy alone.

## Prediction point

Predictions are made at:

`order_approved_at`

Only information available at or before this timestamp may enter the final model feature matrix.

## Leakage

The following must never enter `X`:

- `order_delivered_carrier_date`
- `order_delivered_customer_date`
- `delivery_delay_days`
- review-related information

## Output

The cleaned order-level dataset is written to:

`data/interim/orders_cleaned.parquet`

## Next step

Run `02_eda.ipynb` using this regenerated interim dataset.
